# Gold Layer — Data Warehouse & Time Series Registry

**Objective:** Promote the silver tables to a production-ready data warehouse,
aggregate the calendar to weekly granularity, and build a time series registry
(`dim_time_series`) that assigns a unique ID to every active (supplier, product, region)
combination. Empty series (zero sales across all weeks) are excluded.

**Input :** `data/silver/*.parquet`  
**Output:** `data/gold/data_warehouse/dw_*.parquet`

In [ ]:
import pandas as pd
import numpy as np


In [ ]:
# 1. Load silver tables
df_dim_region   = pd.read_parquet('../../data/silver/dim_region.parquet')
df_dim_product  = pd.read_parquet('../../data/silver/dim_product.parquet')
df_dim_calendar = pd.read_parquet('../../data/silver/dim_calendar.parquet')
df_dim_supplier = pd.read_parquet('../../data/silver/dim_supplier.parquet')
df_fact_sales   = pd.read_parquet('../../data/silver/fact_sales_weekly.parquet')
print("Silver tables loaded.")


## Promote dimension tables to gold

Region, product, and supplier dimensions require no transformation — they are written as-is to the gold layer.


In [ ]:
df_dim_region.to_parquet('../../data/gold/data_warehouse/dw_dim_region.parquet', index=False)
df_dim_product.to_parquet('../../data/gold/data_warehouse/dw_dim_product.parquet', index=False)
df_dim_supplier.to_parquet('../../data/gold/data_warehouse/dw_dim_supplier.parquet', index=False)
print('dim_region / dim_product / dim_supplier  — Saved')


## dim_weekly_calendar

Collapse the daily calendar to weekly granularity: one row per ISO week.
Day-level attributes are dropped; year, semester, quarter, and month attributes
are kept using the first day of each week as representative.


In [ ]:
# Aggregate daily → weekly, drop day-level columns
columns = df_dim_calendar.columns[1:]
df_dim_weekly_calendar = (
    df_dim_calendar
    .groupby('week_date', as_index=False)
    .agg(
        start_date=('date', 'min'),
        end_date=('date', 'max'),
        **{col: (col, 'first') for col in columns}
    )
    .drop(columns=[col for col in columns if 'day' in col])
)

# Reorder: week_date + week first, then remaining temporal attributes
weekly_columns = ['week_date', 'week'] + [
    c for c in df_dim_weekly_calendar.columns if c not in ('week_date', 'week')
]
df_dim_weekly_calendar = df_dim_weekly_calendar[[c for c in weekly_columns if c in df_dim_weekly_calendar.columns]]

df_dim_weekly_calendar.to_parquet('../../data/gold/data_warehouse/dw_dim_weekly_calendar.parquet', index=False)
print(f"dim_weekly_calendar: {len(df_dim_weekly_calendar)} weeks  — Saved")
df_dim_weekly_calendar.head()


## dim_time_series — Time Series Registry

Builds one row per active (supplier, product, region) combination.
For each series, a complete weekly spine is created (filling gaps with zeros)
to compute statistics that correctly account for zero-demand weeks.
Series with zero sales in every week are excluded — they carry no signal.


In [ ]:
def create_time_series_registry(df_fact_sales: pd.DataFrame) -> pd.DataFrame:
    """
    Build a registry of active time series with summary statistics.
    A complete weekly spine is constructed per series to ensure
    zero-demand weeks are counted correctly in all metrics.
    """
    results = []
    groups = df_fact_sales.groupby(["supplier_id", "region_id", "product_id"])
    print(f"Building registry for {groups.ngroups:,} series...")

    for i, ((supplier, region, product), group) in enumerate(groups):
        if (i + 1) % 500 == 0:
            print(f"  {i+1}/{groups.ngroups}")

        first_week = group["week_date"].min()
        last_week  = group["week_date"].max()

        # Full weekly spine for the active period
        spine = pd.DataFrame({
            "supplier_id": supplier, "region_id": region, "product_id": product,
            "week_date": pd.date_range(first_week, last_week, freq="W-MON"),
        })
        series = spine.merge(group[["week_date", "units_sold"]], on="week_date", how="left")
        series["units_sold"] = series["units_sold"].fillna(0)
        units = series["units_sold"].values

        results.append({
            "supplier_id": supplier,
            "region_id":   region,
            "product_id":  product,
            "first_week_date":    first_week,
            "last_week_date":     last_week,
            "total_weeks_length": len(units),
            "num_week_with_sales": int((units > 0).sum()),
            "num_week_with_zeros": int((units == 0).sum()),
            "sales_units":         units.sum(),
            "avg_weekly_sales":    units.mean(),
            "std_weekly_sales":    units.std(ddof=1) if len(units) > 1 else 0,
            "max_weekly_sales":    units.max(),
            "min_weekly_sales":    units.min(),
            "q25_sales":  np.quantile(units, 0.25),
            "q50_sales":  np.quantile(units, 0.50),
            "q75_sales":  np.quantile(units, 0.75),
        })

    df = pd.DataFrame(results)
    # Derived features
    df["sales_weeks_ratio"] = df["num_week_with_sales"] / df["total_weeks_length"]
    df["cv"]  = df["std_weekly_sales"] / (df["avg_weekly_sales"] + 1e-6)
    df["iqr"] = df["q75_sales"] - df["q25_sales"]
    return df

df_dim_time_series = create_time_series_registry(df_fact_sales)
print(f"Registry built: {len(df_dim_time_series):,} series")


In [ ]:
# Inspect and remove series with zero sales across all weeks
empty = (df_dim_time_series["num_week_with_sales"] == 0).sum()
print(f"Empty series (all zeros): {empty} ({empty / len(df_dim_time_series):.1%})")

df_dim_time_series = df_dim_time_series[df_dim_time_series["num_week_with_sales"] > 0].copy()
print(f"Active series retained: {len(df_dim_time_series):,}")


In [ ]:
# Assign surrogate time_series_id and save
df_dim_time_series = df_dim_time_series.reset_index(drop=True)
df_dim_time_series.insert(0, "time_series_id", df_dim_time_series.index + 1)

df_dim_time_series.to_parquet('../../data/gold/data_warehouse/dw_dim_time_series.parquet', index=False)
print(f"dim_time_series: {len(df_dim_time_series):,} series  — Saved")
df_dim_time_series.head()


## dw_fact_sales_weekly

Join the fact table with the time series registry to attach `time_series_id`.
An inner join naturally excludes any (supplier, product, region) combination
not present in the registry (i.e. empty series).


In [ ]:
(
    df_fact_sales
    .merge(df_dim_time_series[["supplier_id", "region_id", "product_id", "time_series_id"]],
           on=["supplier_id", "region_id", "product_id"], how="inner")
    [["week_date", "supplier_id", "region_id", "product_id", "time_series_id", "units_sold"]]
    .to_parquet('../../data/gold/data_warehouse/dw_fact_sales_weekly.parquet', index=False)
)
print('dw_fact_sales_weekly  — Saved')
